# German-French Spread Model Training and Validation

**Research question:** How well does the baseline XGBoost spread model fit the
configured training window, and how does it behave on the later, held-out
validation window?

This notebook is a presentation wrapper around the training API under
`energy_trading_pipeline.models`. It trains through `train_baseline`, the same
function the `train` CLI command runs, so the notebook and the command cannot
drift apart. Everything after training is inspection: the saved artifact is
reloaded, its recorded metrics are reproduced, and its validation behaviour is
plotted.

## Reproducibility boundary

The default `real` mode uses the 2022-2025 SMARD prices and Open-Meteo
ERA5 country weather already cached under `data/processed/`, configured by
`configs/experiment_2022_2025.yaml`, `configs/local_paths_2022_2025.yaml`, and
`configs/model_params.yaml`. It builds the feature dataset and writes it to
`data/features/feature_dataset_2022_2025.parquet` (replaced on every run, with a
`.metadata.yaml` sidecar) so the `train` CLI can use the same file. Set
`ETP_NOTEBOOK_MODE=fixture` to run the same stages on the small repository test
fixtures, writing only to a temporary directory. Neither mode downloads data,
uses credentials, or reaches the network.

The model, its registry entry, and the run log produced here are written to a
temporary directory. Running the notebook never adds a model to
`models/registry/models_index.yaml` or a run to `logs/`; for a recorded run use

```bash
uv run python -m energy_trading_pipeline.cli train \
  --config configs/experiment_2022_2025.yaml \
  --paths-config configs/local_paths_2022_2025.yaml \
  --model-params-config configs/model_params.yaml
```

The final guard cell asserts that the canonical registry is unchanged.

In [ ]:
import copy
import hashlib
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import mean_absolute_error, mean_squared_error

from energy_trading_pipeline.config.loader import load_config
from energy_trading_pipeline.data_ingestion.local_loader import load_local_data
from energy_trading_pipeline.features.dataset_builder import save_feature_dataset
from energy_trading_pipeline.features.lag_features import build_lag_features
from energy_trading_pipeline.models.trainer import (
    TrainingWindow,
    select_training_window,
    train_baseline,
)
from energy_trading_pipeline.models.xgboost_model import XGBoostSpreadModel
from energy_trading_pipeline.preprocessing.alignment import align_hourly_data
from energy_trading_pipeline.preprocessing.cleaning import clean_records
from energy_trading_pipeline.preprocessing.spread import calculate_spread
from energy_trading_pipeline.preprocessing.timestamps import normalize_timestamps

working_directory = Path.cwd()
if (working_directory / "pyproject.toml").is_file():
    REPO_ROOT = working_directory
elif (working_directory.parent / "pyproject.toml").is_file():
    REPO_ROOT = working_directory.parent
else:
    raise FileNotFoundError(
        "Run this notebook from the repository root or notebooks directory"
    )
MODE = os.environ.get("ETP_NOTEBOOK_MODE", "real").strip().lower()
if MODE not in {"real", "fixture"}:
    raise ValueError("ETP_NOTEBOOK_MODE must be 'real' or 'fixture'")
PREVIEW_ROWS = int(os.environ.get("ETP_NOTEBOOK_PREVIEW_ROWS", "5"))

REAL_CONFIG_PATHS = {
    "experiment": REPO_ROOT / "configs/experiment_2022_2025.yaml",
    "local_paths": REPO_ROOT / "configs/local_paths_2022_2025.yaml",
    "model_params": REPO_ROOT / "configs/model_params.yaml",
}
FIXTURE_CONFIG_PATHS = {
    "experiment": REPO_ROOT / "tests/fixtures/sample_config.yaml",
    "model_params": REPO_ROOT / "configs/model_params.yaml",
}
CANONICAL_REGISTRY_INDEX = REPO_ROOT / "models/registry/models_index.yaml"
CANONICAL_RUNS_DIR = REPO_ROOT / "logs/runs"


def show(value):
    # Keep notebook display optional for the pure-Python fixture smoke test.
    try:
        get_ipython
    except NameError:
        if isinstance(value, pd.DataFrame):
            print(value.to_string())
        else:
            print(value)
    else:
        from IPython.display import display
        display(value)


def preview(label, frame, rows=None):
    # Print the shape and leading rows of a stage dataframe.
    row_count = PREVIEW_ROWS if rows is None else rows
    print(f"\n{label}: {len(frame):,} rows x {len(frame.columns)} columns")
    show(frame.head(row_count))


def regression_metrics(actual, prediction):
    # The same RMSE and MAE definitions train_baseline records.
    return {
        "rmse": float(mean_squared_error(actual, prediction) ** 0.5),
        "mae": float(mean_absolute_error(actual, prediction)),
    }


def canonical_state():
    # Fingerprint of the canonical registry and run logs, checked at the end.
    index_digest = (
        hashlib.sha256(CANONICAL_REGISTRY_INDEX.read_bytes()).hexdigest()
        if CANONICAL_REGISTRY_INDEX.is_file()
        else None
    )
    runs = (
        sorted(path.name for path in CANONICAL_RUNS_DIR.iterdir())
        if CANONICAL_RUNS_DIR.is_dir()
        else []
    )
    return {"registry_index_sha256": index_digest, "log_runs": runs}


canonical_state_before = canonical_state()
workspace = tempfile.TemporaryDirectory(prefix="etp_training_notebook_")
WORKSPACE_DIR = Path(workspace.name).resolve()
print(f"Execution mode: {MODE}")
print(f"Temporary workspace: {WORKSPACE_DIR}")

## Stage 1 - Configuration

The experiment config supplies the chronological train and validation date
ranges; the parameters file supplies the XGBoost hyperparameters. Both ranges
are end-inclusive UTC calendar days, and training must end strictly before
validation starts. `fixture` mode reads the test fixture experiment config with
the same model parameters.

In [ ]:
if MODE == "real":
    CONFIG_PATH = REAL_CONFIG_PATHS["experiment"]
    config = load_config(
        CONFIG_PATH,
        local_paths_config_path=REAL_CONFIG_PATHS["local_paths"],
        model_params_config_path=REAL_CONFIG_PATHS["model_params"],
    )
else:
    CONFIG_PATH = FIXTURE_CONFIG_PATHS["experiment"]
    config = load_config(
        CONFIG_PATH,
        model_params_config_path=FIXTURE_CONFIG_PATHS["model_params"],
    )

window = TrainingWindow.from_config(config["dates"])
show(
    pd.DataFrame(
        [{"setting": key, "value": value} for key, value in window.as_metadata().items()]
        + [
            {"setting": "target_column", "value": config["model"]["target_column"]},
            {"setting": "retraining.strategy (recorded only)", "value": config["retraining"]["strategy"]},
        ]
    )
)
print("\nXGBoost parameters")
show(
    pd.DataFrame(
        [{"parameter": key, "value": value} for key, value in config["model"]["params"].items()]
    )
)

## Stage 2 - Feature dataset with lagged weather

The feature dataset is built here from processed hourly data with the
repository feature API, then written with `save_feature_dataset`.

- `real`: the cached 2022-2025 price and country-weather artifacts are required
  to share exactly the same 35,064 UTC timestamps, joined one-to-one, and given
  the `spread` target with `calculate_spread`.
- `fixture`: the fixture prices and weather are cleaned and aligned as in
  notebooks 01 and 02. Fixture weather columns with gaps are excluded rather
  than imputed.

**Weather is lagged, never same-hour.** The weather is ERA5 reanalysis: the
observed weather, published after the fact. Feeding the target hour's wind or
radiation to the model would hand it information no forecaster has, so each
configured weather column is shifted by `features.lag_hours` with
`build_lag_features`. Only the lagged columns become predictors
(`docs/data_sources.md`, ERA5 leakage note). Same-hour `price_de` and
`price_fr` are likewise dropped by the feature builder because they reconstruct
the target.

In [ ]:
if MODE == "real":
    source_paths = {
        "prices": config["paths"]["processed_prices_parquet_path"],
        "weather": config["paths"]["processed_weather_parquet_path"],
    }
    loaded = {}
    for dataset, path in source_paths.items():
        loaded[dataset], _ = normalize_timestamps(load_local_data(path), source_timezone="UTC")
    if not pd.DatetimeIndex(loaded["prices"]["timestamp"]).equals(
        pd.DatetimeIndex(loaded["weather"]["timestamp"])
    ):
        raise ValueError("Price and weather timestamps do not match exactly")
    processed = calculate_spread(
        loaded["prices"].merge(loaded["weather"], on="timestamp", how="inner", validate="one_to_one")
    )
    FEATURE_PATH = config["paths"]["feature_data_parquet_path"]
    source_dataset = " + ".join(str(path.relative_to(REPO_ROOT)) for path in source_paths.values())
else:
    fixture_audit_dir = WORKSPACE_DIR / "run_notebook"
    cleaned = {}
    for source, required in (
        ("price_de", ["price_de"]),
        ("price_fr", ["price_fr"]),
        ("weather", []),
    ):
        normalized, _ = normalize_timestamps(
            load_local_data(REPO_ROOT / config["data"][f"{source}_path"])
        )
        cleaned[source], _ = clean_records(
            normalized,
            required_columns=required,
            optional_columns=[
                column for column in normalized.columns if column not in ("timestamp", *required)
            ],
            duplicate_policy="keep_first",
            required_missing_policy="raise",
            run_dir=fixture_audit_dir,
            dataset=source,
        )
    aligned, _ = align_hourly_data(
        cleaned["price_de"],
        cleaned["price_fr"],
        weather=cleaned["weather"],
        start_date=config["dates"]["start_date"],
        end_date=config["dates"]["end_date"],
    )
    processed = calculate_spread(aligned)
    FEATURE_PATH = WORKSPACE_DIR / "features/feature_dataset.parquet"
    source_dataset = "tests/fixtures (notebook fixture mode)"

weather_sources = [
    column for column in config["features"]["weather_columns"] if column in processed.columns
]
unavailable_weather = [
    column for column in config["features"]["weather_columns"] if column not in processed.columns
]
with_weather_lags, weather_lag_report = build_lag_features(
    processed, config["features"]["lag_hours"], columns=weather_sources
)
lagged_weather_columns = weather_lag_report["generated_columns"]
# The weather selector excludes any column with a gap, and the first
# max(lag_hours) rows of a lag have no history. Those warm-up hours are dropped
# here so the lagged weather survives; the price lags then warm up after them.
weather_warmup_rows = int(with_weather_lags[lagged_weather_columns].isna().any(axis=1).sum())
with_weather_lags = with_weather_lags.dropna(subset=lagged_weather_columns).reset_index(drop=True)
FEATURE_PATH, FEATURE_METADATA_PATH = save_feature_dataset(
    with_weather_lags,
    FEATURE_PATH,
    feature_config={**config["features"], "weather_columns": lagged_weather_columns},
    source_dataset=source_dataset,
)

features = pd.read_parquet(FEATURE_PATH)
features["timestamp"] = pd.to_datetime(features["timestamp"], utc=True)
features = features.sort_values("timestamp", kind="stable").reset_index(drop=True)
# Leakage guards: no same-hour price or weather value reaches the model.
assert not {"price_de", "price_fr"}.intersection(features.columns)
assert not set(weather_sources).intersection(features.columns)
assert set(lagged_weather_columns).issubset(features.columns), "Lagged weather was excluded"

target_column = config["model"]["target_column"]
feature_columns = [column for column in features.columns if column not in ("timestamp", target_column)]
print(f"Processed rows: {len(processed):,} ({processed['timestamp'].min().isoformat()} to {processed['timestamp'].max().isoformat()})")
print(f"Weather lag warm-up rows dropped: {weather_warmup_rows}")
print(f"Feature dataset: {FEATURE_PATH}")
print(f"Feature metadata: {FEATURE_METADATA_PATH}")
if unavailable_weather:
    print(f"Configured weather columns unavailable and skipped: {unavailable_weather}")
print("\nWeather predictors (source column -> lagged predictors)")
show(
    pd.DataFrame(
        [
            {
                "source_column (not a predictor)": column,
                "lagged_predictors": ", ".join(
                    f"{column}_lag_{hours}" for hours in config["features"]["lag_hours"]
                ),
            }
            for column in weather_sources
        ]
    )
)
preview("stage 2 - features", features)
print(f"\n{len(feature_columns)} predictors: {', '.join(feature_columns)}")

## Stage 3 - Chronological train/validation split

`select_training_window` selects rows by the configured calendar days and
verifies on the selected rows, not just the declared bounds, that the last
training timestamp is earlier than the first validation timestamp. The model
never sees a row it is later scored on.

In [ ]:
split = select_training_window(features, window, target_column=target_column)
assert split.train_timestamps.max() < split.validation_timestamps.min()
assert split.feature_columns == feature_columns

show(
    pd.DataFrame(
        [
            {
                "range": name,
                "rows": split.metadata[f"{name}_rows"],
                "first_timestamp": split.metadata[f"{name}_range"]["start"],
                "last_timestamp": split.metadata[f"{name}_range"]["end"],
            }
            for name in ("train", "validation")
        ]
    )
)

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(features["timestamp"], features[target_column], color="0.6", linewidth=1, label="spread (unused)")
ax.plot(split.train_timestamps, split.train_target, color="tab:blue", linewidth=1.2, label="training rows")
ax.plot(split.validation_timestamps, split.validation_target, color="tab:orange", linewidth=1.2, label="validation rows")
ax.axvline(split.validation_timestamps.min(), color="black", linestyle="--", linewidth=1)
ax.set_title("Chronological split of the spread target")
ax.set_ylabel("spread (EUR/MWh)")
ax.legend(loc="upper left")
fig.tight_layout()
plt.show()
plt.close(fig)

## Stage 4 - Training with the CLI training function

`train_baseline` is called with a copy of the configuration whose `models_dir`
and `logs_dir` point into the temporary workspace. It fits only the training
rows, scores the validation rows, saves `model.json` and `metadata.yaml` into
a registry, and writes `run_metadata.yaml`, exactly as the `train` command
does.

In [ ]:
training_config = copy.deepcopy(config)
training_config["paths"] = {
    "feature_data_parquet_path": FEATURE_PATH,
    "models_dir": WORKSPACE_DIR / "models",
    "logs_dir": WORKSPACE_DIR / "logs",
}
run_metadata = train_baseline(training_config, config_file_path=CONFIG_PATH)

show(
    pd.DataFrame(
        [
            {"field": "run_id", "value": run_metadata["run_id"]},
            {"field": "model_version", "value": run_metadata["model_version"]},
            {"field": "validation rmse", "value": run_metadata["metrics"]["rmse"]},
            {"field": "validation mae", "value": run_metadata["metrics"]["mae"]},
            {"field": "model artifact", "value": run_metadata["artifact_paths"]["model"]},
            {"field": "run metadata", "value": run_metadata["artifact_paths"]["run_metadata"]},
        ]
    )
)

## Stage 5 - Reload the artifact and reproduce its metrics

The saved `model.json` is reloaded with `XGBoostSpreadModel.load`. Its stored
feature contract must match the split, and its predictions on the validation
rows must reproduce the RMSE and MAE recorded in the run metadata. Training-row
(in-sample) metrics are shown beside them: a large gap indicates overfitting.

In [ ]:
model = XGBoostSpreadModel.load(Path(run_metadata["artifact_paths"]["model"]))
assert model.feature_columns == split.feature_columns
assert model.params == run_metadata["model_parameters"]

train_prediction = model.predict(split.train_features)
validation_prediction = model.predict(split.validation_features)
reproduced_metrics = regression_metrics(split.validation_target, validation_prediction)
for name, value in run_metadata["metrics"].items():
    assert np.isclose(reproduced_metrics[name], value), (name, reproduced_metrics[name], value)

validation_results = pd.DataFrame(
    {
        "timestamp": split.validation_timestamps,
        "actual": split.validation_target.to_numpy(),
        "prediction": validation_prediction,
    }
)
validation_results["error"] = validation_results["prediction"] - validation_results["actual"]
validation_results["squared_error"] = validation_results["error"] ** 2
validation_results["absolute_error"] = validation_results["error"].abs()

metric_summary = pd.DataFrame(
    [
        {"rows": "training (in-sample)", **regression_metrics(split.train_target, train_prediction)},
        {"rows": "validation (recorded)", **run_metadata["metrics"]},
        {"rows": "validation (reproduced)", **reproduced_metrics},
    ]
)
show(metric_summary)
preview("stage 5 - validation_results", validation_results)

## Stage 6 - Reference baselines

A model score means little without a naive comparison. Two persistence
forecasts are read straight from existing lag features on the same validation
rows: the spread one hour earlier (`spread_lag_1`) and the spread at the same
hour one day earlier (`spread_lag_24`). Only baselines whose lag column is in
the feature set are shown. These are context for this notebook only, not new
pipeline metrics.

In [ ]:
validation_rows = features["timestamp"].isin(split.validation_timestamps)
baseline_rows = [{"forecast": f"xgboost {run_metadata['model_version']}", **reproduced_metrics}]
for column, label in (
    ("spread_lag_1", "persistence: previous hour"),
    ("spread_lag_24", "persistence: same hour yesterday"),
):
    if column in features.columns:
        baseline_rows.append(
            {
                "forecast": label,
                **regression_metrics(
                    features.loc[validation_rows, target_column],
                    features.loc[validation_rows, column],
                ),
            }
        )
baseline_comparison = pd.DataFrame(baseline_rows)
show(baseline_comparison)

## Stage 7 - Validation behaviour

Top: actual versus predicted spread across both windows, with the fitted
training rows shown faintly. Bottom left: predicted against actual on the
validation rows, where a perfect model lies on the diagonal. Bottom right:
the validation error distribution, `error = prediction - actual`.

In [ ]:
fig = plt.figure(figsize=(11, 7.5))
grid = fig.add_gridspec(2, 2, height_ratios=[1.1, 1])

ax_time = fig.add_subplot(grid[0, :])
ax_time.plot(split.train_timestamps, split.train_target, color="0.35", linewidth=1, label="actual")
ax_time.plot(split.train_timestamps, train_prediction, color="tab:blue", linewidth=1, alpha=0.45, label="fitted (training)")
ax_time.plot(validation_results["timestamp"], validation_results["actual"], color="0.35", linewidth=1)
ax_time.plot(validation_results["timestamp"], validation_results["prediction"], color="tab:orange", linewidth=1.4, label="predicted (validation)")
ax_time.axvline(split.validation_timestamps.min(), color="black", linestyle="--", linewidth=1)
ax_time.set_title("Spread: actual vs model")
ax_time.set_ylabel("spread (EUR/MWh)")
ax_time.legend(loc="upper left")

ax_scatter = fig.add_subplot(grid[1, 0])
ax_scatter.scatter(validation_results["actual"], validation_results["prediction"], s=14, color="tab:orange")
limits = [
    min(validation_results["actual"].min(), validation_results["prediction"].min()),
    max(validation_results["actual"].max(), validation_results["prediction"].max()),
]
ax_scatter.plot(limits, limits, color="black", linewidth=1)
ax_scatter.set_title("Validation: predicted vs actual")
ax_scatter.set_xlabel("actual")
ax_scatter.set_ylabel("prediction")

ax_hist = fig.add_subplot(grid[1, 1])
ax_hist.hist(validation_results["error"], bins=20, color="tab:orange", edgecolor="white")
ax_hist.axvline(0, color="black", linewidth=1)
ax_hist.set_title("Validation error distribution")
ax_hist.set_xlabel("error (EUR/MWh)")

fig.tight_layout()
plt.show()
plt.close(fig)

print("Largest absolute validation errors")
show(validation_results.nlargest(PREVIEW_ROWS, "absolute_error"))

## Stage 8 - Feature importance

Gain importance from the fitted booster: the average loss reduction each
feature contributes when it is used in a split. Predictors never used in a
split score zero. Importance describes what this fitted model relies on; it is
not a causal statement about the market.

In [ ]:
gain = model.regressor.get_booster().get_score(importance_type="gain")
feature_importance = (
    pd.DataFrame(
        {
            "feature": model.feature_columns,
            "gain": [float(gain.get(column, 0.0)) for column in model.feature_columns],
        }
    )
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)
feature_importance["share"] = feature_importance["gain"] / feature_importance["gain"].sum()
show(feature_importance)

fig, ax = plt.subplots(figsize=(8, 0.32 * len(feature_importance) + 1))
ordered = feature_importance.iloc[::-1]
ax.barh(ordered["feature"], ordered["share"], color="tab:blue")
ax.set_title("Gain importance share")
ax.set_xlabel("share of total gain")
fig.tight_layout()
plt.show()
plt.close(fig)

## Stage 9 - Saved models in the registry (read-only)

The model registry index is read, never written. In `real` mode this is the
canonical `models/registry/models_index.yaml`; in `fixture` mode it is the
temporary registry from Stage 4. A saved model is rescored on this notebook's
validation rows only when its training window ended before validation starts
and every feature it was trained on is present. Otherwise the reason is shown.

In [ ]:
registry_index_path = (
    CANONICAL_REGISTRY_INDEX
    if MODE == "real"
    else WORKSPACE_DIR / "models/registry/models_index.yaml"
)
registry_index = (
    yaml.safe_load(registry_index_path.read_text(encoding="utf-8")) or {}
    if registry_index_path.is_file()
    else {}
)
registry_rows = []
for version, entry in (registry_index.get("models") or {}).items():
    saved_window = entry.get("training_window") or {}
    recorded = entry.get("validation_metrics") or {}
    saved_features = entry.get("feature_columns") or []
    row = {
        "model_version": version,
        "train_end_date": saved_window.get("train_end_date"),
        "recorded_rmse": recorded.get("rmse"),
        "recorded_mae": recorded.get("mae"),
        "rmse_here": None,
        "mae_here": None,
        "status": "",
    }
    missing_features = [column for column in saved_features if column not in features.columns]
    model_path = Path((entry.get("artifact_paths") or {}).get("model", ""))
    if not saved_window.get("train_end_date") or (
        pd.Timestamp(saved_window["train_end_date"]).date() >= window.validation_start_date
    ):
        row["status"] = "skipped: training window reaches validation rows"
    elif missing_features:
        row["status"] = f"skipped: missing features {missing_features}"
    elif not model_path.is_file():
        row["status"] = "skipped: model artifact not found"
    else:
        saved_model = XGBoostSpreadModel.load(model_path)
        scored = regression_metrics(
            split.validation_target,
            saved_model.predict(features.loc[validation_rows].reset_index(drop=True)),
        )
        row.update(rmse_here=scored["rmse"], mae_here=scored["mae"], status="rescored")
    registry_rows.append(row)

print(f"Registry index: {registry_index_path}")
registry_summary = pd.DataFrame(registry_rows)
show(registry_summary if not registry_summary.empty else "No saved models are registered.")

## Guard - canonical artifacts unchanged

The fingerprint taken in the setup cell is compared with the current state:
the canonical registry index and the `logs/runs` listing must be identical.

In [ ]:
canonical_state_after = canonical_state()
assert canonical_state_after == canonical_state_before, "Notebook wrote to canonical models/ or logs/"
print("Canonical registry and run logs unchanged.")

## Limitations and handoff

- **One-step-ahead, not day-ahead.** The feature set includes 1-hour lags
  (`spread_lag_1`, `price_de_lag_1`, `price_fr_lag_1`) and 24-hour rolling
  statistics computed up to the previous hour. Each validation row is scored
  with the true previous hour available, so the RMSE above measures
  one-hour-ahead accuracy. A day-ahead forecast issued once per day does not
  know most of those values. The `backtest` command enforces data availability
  at issuance and is the source of forecast-accuracy results for the research
  question.
- **Lagged weather understates the weather effect.** The spread responds to
  the wind and solar of the delivery hour, which a real forecaster knows only
  as a day-ahead weather forecast. Lagged ERA5 is the leakage-safe stand-in
  available today; archived day-ahead weather forecasts (forecast vintages)
  would be the faithful predictor and would likely carry more signal.
- **Regime changes inside the data.** Training spans the 2022 energy crisis
  (French nuclear outages, French prices up to about 2,990 EUR/MWh) as well as
  the calmer 2023-2024 market, and validation covers 2025, including the move to
  15-minute day-ahead products on 1 October 2025. From that date the hourly
  price is the mean of four quarter-hour prices.
- **Single split, no tuning.** One chronological split is evaluated with the
  configured parameters. No hyperparameter search or cross-validation is done
  here, and changing parameters should go through `configs/model_params.yaml`
  so every retraining strategy shares them.
- **Temporary artifacts.** Everything trained here is discarded with the
  temporary workspace. Record a run with the `train` CLI command.

The retraining-strategy comparison (`no_retraining`, `fixed_schedule`,
`performance_triggered`) starts from the same model configuration and is run
through backtesting, not this notebook.